In [1]:
import patito as pt
import polars as pl 
from datetime import date
from typing import Literal 

In [2]:
class Order(pt.Model):
    order_id: str
    customer_id: int
    order_date: date
    quantity: int = pt.Field(ge=1)
    price: float = pt.Field(gt=0)
    total: float
    status: Literal["pending", "processing", "completed", "cancelled"]

In [5]:
df = pl.read_csv('data_error.csv')
df

order_id,customer_id,order_date,quantity,price,total,status
str,str,str,i64,f64,f64,str
"""ORD1001""","""123""","""2024-01-10""",2,10.5,21.0,"""completed"""
"""ORD1002""",null,"""2024-02-05""",-1,20.0,-20.0,"""pending"""
"""ORDX003""","""456""","""2024-13-01""",3,15.0,45.0,"""processing"""
null,"""789""","""not_a_date""",5,-5.0,-25.0,"""completed"""
"""ORD1005""","""abc""","""2024-03-20""",0,10.0,0.0,"""shipped"""
"""ORD1006""","""321""","""2024-02-29""",1,12.5,25.0,"""PROCESSING"""
"""ORD1007""","""654""","""2024-04-31""",3,0.0,0.0,"""pending"""
"""ORD1008""","""987""","""2024-06-15""",3,20.0,40.0,"""cancelled"""
"""ORD1009""","""111""","""2024-07-07""",10,100.0,900.0,"""completed"""


In [6]:
try: 
    Order.validate(df)
    print('Xác thực thành công')
except pt.DataFrameValidationError as e:
    print(e)

7 validation errors for Order
customer_id
  1 missing value (type=value_error.missingvalues)
order_id
  1 missing value (type=value_error.missingvalues)
customer_id
  Polars dtype String does not match model field type. (type=type_error.columndtype)
order_date
  Polars dtype String does not match model field type. (type=type_error.columndtype)
quantity
  2 rows with out of bound values. (type=value_error.rowvalue)
price
  2 rows with out of bound values. (type=value_error.rowvalue)
status
  Rows with invalid values: {'shipped', 'PROCESSING'}. (type=value_error.rowvalue)


In [17]:
error_rows = []
valid_rows = []

for row in df.to_dicts():
    try:
        validated = Order.model_validate(row)
        valid_rows.append(validated.model_dump())
    except Exception as e:
        error_rows.append({**row, "error": str(e)})


In [9]:
valid_df = pl.DataFrame(valid_rows)
valid_df

order_id,customer_id,order_date,quantity,price,total,status
str,i64,date,i64,f64,f64,str
"""ORD1001""",123,2024-01-10,2,10.5,21.0,"""completed"""
"""ORD1008""",987,2024-06-15,3,20.0,40.0,"""cancelled"""
"""ORD1009""",111,2024-07-07,10,100.0,900.0,"""completed"""
"""ORD1010""",222,2024-05-10,2,8.0,15.0,"""pending"""


In [11]:
error_df =pl.DataFrame(error_rows)
error_df


order_id,customer_id,order_date,quantity,price,total,status,error
str,str,str,i64,f64,f64,str,str
"""ORD1002""",null,"""2024-02-05""",-1,20.0,-20.0,"""pending""","""2 validation errors for Order …"
"""ORDX003""","""456""","""2024-13-01""",3,15.0,45.0,"""processing""","""1 validation error for Order o…"
null,"""789""","""not_a_date""",5,-5.0,-25.0,"""completed""","""3 validation errors for Order …"
"""ORD1005""","""abc""","""2024-03-20""",0,10.0,0.0,"""shipped""","""3 validation errors for Order …"
"""ORD1006""","""321""","""2024-02-29""",1,12.5,25.0,"""PROCESSING""","""1 validation error for Order s…"
"""ORD1007""","""654""","""2024-04-31""",3,0.0,0.0,"""pending""","""2 validation errors for Order …"


In [25]:
gold_df = valid_df.with_columns(
    (pl.col("price") * pl.col("quantity")).alias("revenue")
)
gold_df 

order_id,customer_id,order_date,quantity,price,total,status,revenue
str,i64,date,i64,f64,f64,str,f64
"""ORD1001""",123,2024-01-10,2,10.5,21.0,"""completed""",21.0
"""ORD1008""",987,2024-06-15,3,20.0,40.0,"""cancelled""",60.0
"""ORD1009""",111,2024-07-07,10,100.0,900.0,"""completed""",1000.0
"""ORD1010""",222,2024-05-10,2,8.0,15.0,"""pending""",16.0


In [27]:
error_df.write_csv('filterd_data.csv')

In [28]:
gold_df.write_csv('clean_data.csv')